# Population density per municipality

This notebook processes the 2024 municipality-level dataset and calculates the natural logarithm of population density.

**Input (`data/raw/`):** `kwb2024_per_gemeente.csv`  
**Output (`data/processed/`):** `population_density_per_municipality.csv`

`population_density` is taken directly from `bevolkingsdichtheid` (inhabitants per km²) in the source data. `log_population_density` is the natural logarithm, rounded to four decimal places.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
# Resolve paths (from the notebooks folder or the repository root)
root = Path.cwd()
if not (root / "data" / "raw").is_dir():
    root = root.parent

raw_path = root / "data" / "raw" / "kwb2024_per_gemeente.csv"
output_path = root / "data" / "processed" / "population_density_per_municipality.csv"

raw = pd.read_csv(raw_path)
print("Raw rows:", len(raw))
raw.head()

Raw rows: 342


,gemeentecode,gemeente,opl_laag,opl_middel,opl_hoog,bevolkingsdichtheid,meergezinswoning_pct
0,GM0014,Groningen,31740.0,78560,89290,1314,61
1,GM0034,Almere,47430.0,74710,54260,1753,30
2,GM0037,Stadskanaal,7480.0,11820,4090,272,21
3,GM0047,Veendam,6330.0,9920,4280,362,21
4,GM0050,Zeewolde,4200.0,8570,5890,97,18


In [3]:
# Select the original population density field
population = raw[["gemeente", "bevolkingsdichtheid"]].copy()
population.columns = ["municipality", "population_density"]

# Standardize the seven municipality names used in the processed datasets
name_corrections = {
    "'s-Gravenhage": "Den Haag",
    "Beek (L.)": "Beek",
    "Hengelo (O.)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk",
    "Stein (L.)": "Stein",
}
population["municipality"] = population["municipality"].replace(name_corrections)

# Calculate natural logarithm (ln) of inhabitants per km²
population["population_density"] = pd.to_numeric(
    population["population_density"], errors="raise"
)
if population["population_density"].isna().any() or (population["population_density"] <= 0).any():
    raise ValueError("Population density must be present and positive for the logarithm.")

population["log_population_density"] = np.log(
    population["population_density"]
).round(4)

# Check the data before saving
assert len(population) == 342, "Unexpected number of municipalities"
assert population["municipality"].is_unique, "Duplicate municipality names"
assert not population.isna().any().any(), "Missing values in processed data"

population.head()

,municipality,population_density,log_population_density
0,Groningen,1314,7.1808
1,Almere,1753,7.4691
2,Stadskanaal,272,5.6058
3,Veendam,362,5.8916
4,Zeewolde,97,4.5747


In [4]:
# Save with exactly the existing filename and column order
output_path.parent.mkdir(parents=True, exist_ok=True)
population.to_csv(output_path, index=False)
print(f"Saved {len(population)} municipalities to {output_path}")
population.head()

Saved 342 municipalities to /Users/merelvos/Documents/GitHub/group-15/data/processed/population_density_per_municipality.csv


,municipality,population_density,log_population_density
0,Groningen,1314,7.1808
1,Almere,1753,7.4691
2,Stadskanaal,272,5.6058
3,Veendam,362,5.8916
4,Zeewolde,97,4.5747
